# Demo 2 - Learning Version: Document Classification with Docling and LangChain

Welcome! This interactive notebook will teach you how to build a document classification system using Docling for content extraction and LangChain for structured classification.

## Learning Objectives
By the end of this notebook, you will:
1. Extract and categorize content from documents using Docling
2. Build structured classification pipelines with LangChain and Pydantic
3. Implement batch processing for multiple documents
4. Store and query results using DuckDB/DuckLake
5. Evaluate classification accuracy and generate metrics

## Key Technologies
- **Docling**: Automatic document parsing and element extraction
- **LangChain**: Structured LLM outputs for classification
- **Pydantic**: Type-safe schema definitions
- **DuckDB**: Analytics database for storing results

## Structure
Each section contains:
- **Learning Context**: What you'll learn
- **Task**: An exercise to complete
- **Hints**: Helpful guidance
- **Solution**: Complete implementation (in a separate cell)

Let's get started! 🚀

## Setup: Import Required Libraries

First, let's import all the libraries we'll need. These are already provided for you.

In [ ]:
!pip install docling langchain-core langchain-openai langchain-anthropic duckdb pydantic

In [ ]:
# Import required libraries
import os
import json
from pathlib import Path
from typing import List, Dict, Any, Optional
from datetime import datetime
import pandas as pd
from enum import Enum

# Docling imports
from docling.document_converter import DocumentConverter, PdfFormatOption, WordFormatOption, PowerpointFormatOption, HTMLFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.chunking import HybridChunker
from docling_core.types.doc import (
    TextItem, 
    TableItem, 
    PictureItem,
    ListItem,
    CodeItem,
    FormulaItem,
    SectionHeaderItem
)

# LangChain imports
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, model_validator

# DuckDB for storage
import duckdb

# Load environment variables
import dotenv
dotenv.load_dotenv()

print("✅ All imports successful")

## Task 1: Create a Docling Content Extractor

### Learning Context
Docling automatically categorizes document content into different types:
- **Text elements**: Regular paragraphs, headings
- **Tables**: Structured tabular data
- **Figures**: Images with captions
- **Code blocks**: Programming code snippets
- **Formulas**: Mathematical expressions
- **Lists**: Bulleted or numbered lists
- **Sections**: Document structure

### Your Task
Complete the `DoclingContentExtractor` class that:
1. Initializes a DocumentConverter with proper options
2. Extracts different content types from documents
3. Organizes content by category
4. Returns structured content with statistics

### Hints
- Use `PdfPipelineOptions` to configure extraction features
- Iterate through document items with `doc.iterate_items()`
- Use `isinstance()` to check item types
- Each item type has specific properties (text, caption, etc.)

In [ ]:
class DoclingContentExtractor:
    """Extract and categorize content from documents using Docling"""
    
    def __init__(self, enable_table_structure: bool = True, 
                 enable_picture_classification: bool = True):
        """Initialize Docling converter with specified options"""
        
        # TODO: Step 1 - Configure pipeline options
        # pipeline_options = PdfPipelineOptions(
        #     do_table_structure=enable_table_structure,
        #     do_picture_classification=enable_picture_classification
        # )
        
        # TODO: Step 2 - Configure table extraction mode if enabled
        # if enable_table_structure:
        #     pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
        #     pipeline_options.table_structure_options.do_cell_matching = True
        
        # TODO: Step 3 - Initialize converter with format options
        # self.converter = DocumentConverter(
        #     format_options={
        #         InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
        #         # Add other formats...
        #     }
        # )
        
        pass  # Remove this when you implement
        
    def extract_content(self, file_path: str) -> Dict[str, Any]:
        """Extract all content types from a document"""
        
        # TODO: Step 1 - Convert document
        # result = self.converter.convert(file_path)
        # doc = result.document
        
        # TODO: Step 2 - Initialize content structure
        # content = {
        #     "metadata": self._extract_metadata(doc),
        #     "text_elements": [],
        #     "tables": [],
        #     "figures": [],
        #     "code_blocks": [],
        #     "formulas": [],
        #     "lists": [],
        #     "sections": []
        # }
        
        # TODO: Step 3 - Iterate through document elements
        # for item, level in doc.iterate_items():
        #     if isinstance(item, TextItem):
        #         # Add text element
        #     elif isinstance(item, TableItem):
        #         # Add table
        #     # Add other item types...
        
        # TODO: Step 4 - Add statistics
        # content["statistics"] = {
        #     "total_text_elements": len(content["text_elements"]),
        #     # Add other statistics...
        # }
        
        pass  # Remove this when you implement
    
    def _extract_metadata(self, doc: Any) -> Dict[str, Any]:
        """Extract document metadata"""
        metadata = {}
        
        # TODO: Extract metadata from document
        # if hasattr(doc, 'metadata'):
        #     metadata = ...
        
        # Add extraction timestamp
        metadata["extraction_timestamp"] = datetime.now().isoformat()
        
        return metadata

# Create your extractor instance
# extractor = DoclingContentExtractor(...)

### Solution for Task 1

In [ ]:
# SOLUTION
class DoclingContentExtractor:
    """Extract and categorize content from documents using Docling"""
    
    def __init__(self, enable_table_structure: bool = True, 
                 enable_picture_classification: bool = True):
        """Initialize Docling converter with specified options"""
        
        # Configure pipeline options
        pipeline_options = PdfPipelineOptions(
            do_table_structure=enable_table_structure,
            do_picture_classification=enable_picture_classification
        )
        
        # Use accurate TableFormer mode for better table extraction
        if enable_table_structure:
            pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
            pipeline_options.table_structure_options.do_cell_matching = True
        
        # Initialize converter
        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
                InputFormat.PPTX: PowerpointFormatOption(pipeline_options=pipeline_options),
                InputFormat.DOCX: WordFormatOption(pipeline_options=pipeline_options),
                InputFormat.HTML: HTMLFormatOption(pipeline_options=pipeline_options)
            }
        )
        
    def extract_content(self, file_path: str) -> Dict[str, Any]:
        """Extract all content types from a document"""
        
        # Convert document
        result = self.converter.convert(file_path)
        doc = result.document
        
        # Initialize categorized content
        content = {
            "metadata": self._extract_metadata(doc),
            "text_elements": [],
            "tables": [],
            "figures": [],
            "code_blocks": [],
            "formulas": [],
            "lists": [],
            "sections": []
        }
        
        # Iterate through document elements
        for item, level in doc.iterate_items():
            if isinstance(item, TextItem):
                content["text_elements"].append({
                    "text": item.text,
                    "level": level,
                    "type": "text"
                })
            elif isinstance(item, TableItem):
                content["tables"].append({
                    "dataframe": item.export_to_dataframe(),
                    "caption": getattr(item, 'caption', None),
                    "level": level,
                    "type": "table"
                })
            elif isinstance(item, PictureItem):
                content["figures"].append({
                    "caption": getattr(item, 'caption', None),
                    "annotations": getattr(item, 'annotations', []),
                    "level": level,
                    "type": "figure"
                })
            elif isinstance(item, CodeItem):
                content["code_blocks"].append({
                    "code": item.text,
                    "language": getattr(item, 'language', 'unknown'),
                    "level": level,
                    "type": "code"
                })
            elif isinstance(item, FormulaItem):
                content["formulas"].append({
                    "formula": item.text,
                    "level": level,
                    "type": "formula"
                })
            elif isinstance(item, ListItem):
                content["lists"].append({
                    "items": item.text.split('\n'),
                    "level": level,
                    "type": "list"
                })
            elif isinstance(item, SectionHeaderItem):
                content["sections"].append({
                    "title": item.text,
                    "level": level,
                    "type": "section"
                })
        
        # Add summary statistics
        content["statistics"] = {
            "total_text_elements": len(content["text_elements"]),
            "total_tables": len(content["tables"]),
            "total_figures": len(content["figures"]),
            "total_code_blocks": len(content["code_blocks"]),
            "total_formulas": len(content["formulas"]),
            "total_lists": len(content["lists"]),
            "total_sections": len(content["sections"])
        }
        
        return content
    
    def _extract_metadata(self, doc: Any) -> Dict[str, Any]:
        """Extract document metadata"""
        metadata = {}
        
        # Try to get document properties
        if hasattr(doc, 'metadata'):
            metadata = doc.metadata.export_json_dict() if hasattr(doc.metadata, 'export_json_dict') else {}
        
        # Add extraction timestamp
        metadata["extraction_timestamp"] = datetime.now().isoformat()
        
        return metadata

# Create extractor instance
extractor = DoclingContentExtractor(
    enable_table_structure=True,
    enable_picture_classification=True
)

print("✅ DoclingContentExtractor initialized")

## Task 2: Define Pydantic Classification Schemas

### Learning Context
Pydantic provides type-safe data validation and serialization. When combined with LangChain, it enables structured outputs from LLMs. We'll define schemas for:
- **Slide types**: title, content, chart, table, etc.
- **Content themes**: technical, business, educational, etc.
- **Visual density**: text-heavy, balanced, visual-heavy
- **Classification metadata**: confidence scores, key topics

### Your Task
Define Pydantic schemas for document classification:
1. Create enums for slide types, themes, and visual density
2. Build a `SlideClassification` model with validation
3. Create a `DocumentClassification` model for overall document

### Hints
- Use `Enum` for predefined categories
- Use `Field` to add descriptions and constraints
- Use `@model_validator` for custom validation logic
- Consider min/max values for scores

In [ ]:
# Define classification enums

class SlideType(str, Enum):
    """Types of slides in a presentation"""
    # TODO: Add slide types
    # TITLE = "title"
    # CONTENT = "content"
    # Add more types...
    pass

class ContentTheme(str, Enum):
    """Themes of content"""
    # TODO: Add content themes
    pass

class VisualDensity(str, Enum):
    """Visual density classification"""
    # TODO: Add visual density types
    pass

class SlideClassification(BaseModel):
    """Classification schema for a single slide"""
    
    # TODO: Add fields
    # slide_type: SlideType = Field(
    #     description="The primary type/purpose of this slide"
    # )
    # content_theme: ContentTheme = Field(...)
    # visual_density: VisualDensity = Field(...)
    
    # TODO: Add list fields for topics
    # key_topics: List[str] = Field(
    #     description="List of 3-5 key topics",
    #     min_items=1,
    #     max_items=5
    # )
    
    # TODO: Add boolean fields
    # has_code: bool = Field(...)
    
    # TODO: Add score fields with constraints
    # complexity_score: int = Field(
    #     description="Complexity from 1-10",
    #     ge=1,
    #     le=10
    # )
    
    # TODO: Add validation
    # @model_validator(mode='before')
    # @classmethod
    # def validate_scores(cls, values: dict) -> dict:
    #     """Ensure scores are within valid ranges"""
    #     return values
    
    pass

class DocumentClassification(BaseModel):
    """Overall document classification"""
    
    # TODO: Add document-level fields
    # document_type: str = Field(...)
    # primary_theme: ContentTheme = Field(...)
    # slide_classifications: List[SlideClassification] = Field(...)
    
    pass

print("✅ Pydantic classification schemas defined")

### Solution for Task 2

In [ ]:
# SOLUTION
# Define Pydantic schemas for classification

class SlideType(str, Enum):
    """Types of slides in a presentation"""
    TITLE = "title"
    CONTENT = "content"
    CHART = "chart"
    TABLE = "table"
    CONCLUSION = "conclusion"
    AGENDA = "agenda"
    REFERENCES = "references"
    QUESTIONS = "questions"

class ContentTheme(str, Enum):
    """Themes of content"""
    TECHNICAL = "technical"
    BUSINESS = "business"
    EDUCATIONAL = "educational"
    RESEARCH = "research"
    MARKETING = "marketing"
    GENERAL = "general"

class VisualDensity(str, Enum):
    """Visual density classification"""
    TEXT_HEAVY = "text_heavy"
    BALANCED = "balanced"
    VISUAL_HEAVY = "visual_heavy"
    MINIMAL = "minimal"

class SlideClassification(BaseModel):
    """Classification schema for a single slide"""
    
    slide_type: SlideType = Field(
        description="The primary type/purpose of this slide"
    )
    content_theme: ContentTheme = Field(
        description="The main theme or domain of the content"
    )
    visual_density: VisualDensity = Field(
        description="The visual density based on text vs visual elements ratio"
    )
    key_topics: List[str] = Field(
        description="List of 3-5 key topics or concepts mentioned in the slide",
        min_items=1,
        max_items=5
    )
    has_code: bool = Field(
        description="Whether the slide contains code snippets"
    )
    has_formulas: bool = Field(
        description="Whether the slide contains mathematical formulas"
    )
    has_tables: bool = Field(
        description="Whether the slide contains tables"
    )
    has_figures: bool = Field(
        description="Whether the slide contains figures or images"
    )
    complexity_score: int = Field(
        description="Complexity score from 1 (very simple) to 10 (very complex)",
        ge=1,
        le=10
    )
    confidence_score: float = Field(
        description="Confidence in this classification from 0.0 to 1.0",
        ge=0.0,
        le=1.0
    )
    
    @model_validator(mode='before')
    @classmethod
    def validate_scores(cls, values: dict) -> dict:
        """Ensure scores are within valid ranges"""
        if 'complexity_score' in values:
            values['complexity_score'] = max(1, min(10, values['complexity_score']))
        if 'confidence_score' in values:
            values['confidence_score'] = max(0.0, min(1.0, values['confidence_score']))
        return values

class DocumentClassification(BaseModel):
    """Overall document classification"""
    
    document_type: str = Field(
        description="Type of document (presentation, paper, report, etc.)"
    )
    primary_theme: ContentTheme = Field(
        description="The dominant theme across the document"
    )
    target_audience: str = Field(
        description="Intended audience (students, professionals, researchers, etc.)"
    )
    overall_complexity: int = Field(
        description="Overall complexity from 1 to 10",
        ge=1,
        le=10
    )
    slide_classifications: List[SlideClassification] = Field(
        description="Classifications for individual slides/sections"
    )

print("✅ Pydantic classification schemas defined")

## Task 3: Build LangChain Classification Pipeline

### Learning Context
LangChain enables structured outputs from LLMs using:
- **ChatPromptTemplate**: Define prompts with variables
- **with_structured_output**: Force LLM to return Pydantic models
- **Chain composition**: Connect prompts, LLMs, and parsers

### Your Task
Create a `LangChainDocumentClassifier` that:
1. Initializes an LLM with structured output capability
2. Creates prompts for slide and document classification
3. Implements classification methods
4. Handles content preparation and formatting

### Hints
- Use `ChatAnthropic` or `ChatOpenAI` as the LLM
- Call `llm.with_structured_output(SchemaClass)` for structured outputs
- Create detailed system prompts that guide classification
- Prepare human prompts with extracted content details

In [ ]:
class LangChainDocumentClassifier:
    """Document classifier using LangChain with structured output"""
    
    def __init__(self, model_name: str = "claude-3-5-sonnet-20240620", temperature: float = 0.0):
        """Initialize classifier with specified LLM"""
        
        # TODO: Step 1 - Initialize LLM
        # self.llm = ChatAnthropic(
        #     model_name=model_name,
        #     temperature=temperature
        # )
        
        # TODO: Step 2 - Create structured LLMs
        # self.slide_classifier = self.llm.with_structured_output(SlideClassification)
        # self.document_classifier = self.llm.with_structured_output(DocumentClassification)
        
        # TODO: Step 3 - Create slide classification prompt
        # self.slide_prompt = ChatPromptTemplate.from_messages([
        #     ("system", """You are an expert document analyst..."""),
        #     ("human", """Analyze this slide/section...
        #     
        #     Content Summary: {content_summary}
        #     Text Elements: {text_count}
        #     ...""")
        # ])
        
        # TODO: Step 4 - Create document classification prompt
        # self.document_prompt = ChatPromptTemplate.from_messages([...])
        
        pass
    
    def classify_slide(self, slide_content: Dict[str, Any]) -> SlideClassification:
        """Classify a single slide/section"""
        
        # TODO: Step 1 - Prepare content summary
        # content_summary = self._prepare_slide_summary(slide_content)
        
        # TODO: Step 2 - Prepare prompt inputs
        # prompt_input = {
        #     "content_summary": content_summary,
        #     "text_count": len(slide_content.get("text_elements", [])),
        #     # Add other counts...
        # }
        
        # TODO: Step 3 - Get classification
        # prompt = self.slide_prompt.invoke(prompt_input)
        # classification = self.slide_classifier.invoke(prompt)
        
        # return classification
        
        pass
    
    def _prepare_slide_summary(self, slide_content: Dict[str, Any]) -> str:
        """Prepare a summary of slide content"""
        summary_parts = []
        
        # TODO: Build summary from content counts
        # texts = slide_content.get("text_elements", [])
        # if texts:
        #     summary_parts.append(f"Contains {len(texts)} text elements")
        
        return ". ".join(summary_parts)
    
    def _get_sample_text(self, slide_content: Dict[str, Any]) -> str:
        """Get sample text from slide"""
        # TODO: Extract and combine sample text
        return "No text content"

# Create your classifier instance
# classifier = LangChainDocumentClassifier(...)

### Solution for Task 3

In [ ]:
# SOLUTION
class LangChainDocumentClassifier:
    """Document classifier using LangChain with structured output"""
    
    def __init__(self, model_name: str = "claude-3-5-sonnet-20240620", temperature: float = 0.0):
        """Initialize classifier with specified LLM"""
        
        # Initialize LLM with structured output
        self.llm = ChatAnthropic(
            model_name=model_name,
            temperature=temperature
        )
        
        # Create structured LLMs for different classification tasks
        self.slide_classifier = self.llm.with_structured_output(SlideClassification)
        self.document_classifier = self.llm.with_structured_output(DocumentClassification)
        
        # Create prompts
        self.slide_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert document analyst. Analyze the given slide/section content and classify it according to the provided schema.
            
Consider:
- The primary purpose and type of the slide
- The content theme and domain
- The balance between text and visual elements
- Key topics and concepts
- Presence of special elements (code, formulas, tables, figures)
- Overall complexity

Be precise and confident in your classification."""),
            ("human", """Analyze this slide/section and provide classification:

Content Summary:
{content_summary}

Text Elements: {text_count}
Tables: {table_count}
Figures: {figure_count}
Code Blocks: {code_count}
Formulas: {formula_count}

Sample Text:
{sample_text}

Classify this content.""")
        ])
        
        self.document_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert document analyst. Analyze the entire document and provide an overall classification.
            
Consider:
- Document type and structure
- Primary theme across all content
- Target audience
- Overall complexity
- Individual slide/section classifications

Provide a comprehensive document classification."""),
            ("human", """Analyze this document:

Document Statistics:
{statistics}

Section Titles:
{section_titles}

Content Preview:
{content_preview}

Individual Classifications:
{slide_classifications}

Provide overall document classification.""")
        ])
    
    def classify_slide(self, slide_content: Dict[str, Any]) -> SlideClassification:
        """Classify a single slide/section"""
        
        # Prepare content summary
        content_summary = self._prepare_slide_summary(slide_content)
        
        # Prepare prompt inputs
        prompt_input = {
            "content_summary": content_summary,
            "text_count": len(slide_content.get("text_elements", [])),
            "table_count": len(slide_content.get("tables", [])),
            "figure_count": len(slide_content.get("figures", [])),
            "code_count": len(slide_content.get("code_blocks", [])),
            "formula_count": len(slide_content.get("formulas", [])),
            "sample_text": self._get_sample_text(slide_content)
        }
        
        # Get classification
        prompt = self.slide_prompt.invoke(prompt_input)
        classification = self.slide_classifier.invoke(prompt)
        
        return classification
    
    def classify_document(self, extracted_content: Dict[str, Any], 
                         slide_classifications: List[SlideClassification]) -> DocumentClassification:
        """Classify entire document"""
        
        # Prepare document summary
        statistics = json.dumps(extracted_content.get("statistics", {}), indent=2)
        section_titles = "\n".join([s["title"] for s in extracted_content.get("sections", [])[:10]])
        content_preview = self._get_document_preview(extracted_content)
        
        # Format slide classifications
        slide_class_str = "\n".join([
            f"Slide {i+1}: {cls.slide_type.value} - {cls.content_theme.value}"
            for i, cls in enumerate(slide_classifications[:10])
        ])
        
        # Prepare prompt inputs
        prompt_input = {
            "statistics": statistics,
            "section_titles": section_titles,
            "content_preview": content_preview,
            "slide_classifications": slide_class_str
        }
        
        # Get classification
        prompt = self.document_prompt.invoke(prompt_input)
        
        # Create document classification with slide classifications
        doc_class = self.document_classifier.invoke(prompt)
        doc_class.slide_classifications = slide_classifications
        
        return doc_class
    
    def _prepare_slide_summary(self, slide_content: Dict[str, Any]) -> str:
        """Prepare a summary of slide content"""
        summary_parts = []
        
        # Add text elements
        texts = slide_content.get("text_elements", [])
        if texts:
            summary_parts.append(f"Contains {len(texts)} text elements")
        
        # Add table info
        tables = slide_content.get("tables", [])
        if tables:
            summary_parts.append(f"Contains {len(tables)} tables")
        
        # Add figure info
        figures = slide_content.get("figures", [])
        if figures:
            summary_parts.append(f"Contains {len(figures)} figures")
        
        # Add code info
        code_blocks = slide_content.get("code_blocks", [])
        if code_blocks:
            summary_parts.append(f"Contains {len(code_blocks)} code blocks")
        
        return ". ".join(summary_parts)
    
    def _get_sample_text(self, slide_content: Dict[str, Any]) -> str:
        """Get sample text from slide"""
        texts = slide_content.get("text_elements", [])
        if texts:
            # Combine first few text elements
            sample = " ".join([t["text"] for t in texts[:3]])
            return sample[:500] + "..." if len(sample) > 500 else sample
        return "No text content"
    
    def _get_document_preview(self, extracted_content: Dict[str, Any]) -> str:
        """Get document preview"""
        preview_parts = []
        
        # Add some text samples
        texts = extracted_content.get("text_elements", [])
        if texts:
            preview_parts.append("Text samples:")
            for t in texts[:3]:
                preview_parts.append(f"- {t['text'][:100]}...")
        
        # Add section titles
        sections = extracted_content.get("sections", [])
        if sections:
            preview_parts.append("\nSection titles:")
            for s in sections[:5]:
                preview_parts.append(f"- {s['title']}")
        
        return "\n".join(preview_parts)

# Create classifier instance
classifier = LangChainDocumentClassifier(model_name="claude-3-5-sonnet-20240620", temperature=0.0)
print("✅ LangChainDocumentClassifier initialized")

## Task 4: Implement Batch Processing

### Learning Context
Real-world applications need to process multiple documents efficiently. Batch processing requires:
- Progress tracking
- Error handling for individual documents
- Result aggregation
- Performance optimization

### Your Task
Create a `BatchDocumentProcessor` that:
1. Processes multiple documents with progress updates
2. Handles errors gracefully (don't stop if one fails)
3. Collects statistics and generates summaries
4. Limits processing per document for performance

### Hints
- Track status: "pending", "success", "partial", "failed"
- Use try/except blocks for each document and slide
- Limit slides per document to avoid timeouts
- Generate summary statistics at the end

In [ ]:
class BatchDocumentProcessor:
    """Batch processor for multiple documents"""
    
    def __init__(self, extractor: DoclingContentExtractor, 
                 classifier: LangChainDocumentClassifier):
        """Initialize with extractor and classifier"""
        self.extractor = extractor
        self.classifier = classifier
        self.results = []
        
    def process_documents(self, file_paths: List[str], 
                         max_slides_per_doc: int = 10) -> List[Dict[str, Any]]:
        """Process multiple documents in batch"""
        
        results = []
        total_files = len(file_paths)
        
        print(f"🚀 Starting batch processing of {total_files} documents...")
        
        for idx, file_path in enumerate(file_paths):
            print(f"\n📄 Processing document {idx+1}/{total_files}: {os.path.basename(file_path)}")
            
            # TODO: Step 1 - Initialize result structure
            # result = {
            #     "file_path": file_path,
            #     "file_name": os.path.basename(file_path),
            #     "status": "pending",
            #     "extraction": None,
            #     "classification": None,
            #     "errors": []
            # }
            
            # TODO: Step 2 - Extract content with error handling
            # try:
            #     print("  📊 Extracting content...")
            #     extracted = self.extractor.extract_content(file_path)
            #     result["extraction"] = extracted
            #     print(f"    ✅ Extracted: {extracted['statistics']}")
            # except Exception as e:
            #     # Handle extraction error
            
            # TODO: Step 3 - Classify slides
            # print("  🏷️ Classifying content...")
            # slide_contents = self._prepare_slide_contents(extracted, max_slides_per_doc)
            # slide_classifications = []
            # 
            # for i, slide_content in enumerate(slide_contents):
            #     try:
            #         classification = self.classifier.classify_slide(slide_content)
            #         slide_classifications.append(classification)
            #     except Exception as e:
            #         # Handle classification error
            
            # TODO: Step 4 - Classify document
            # if slide_classifications:
            #     try:
            #         doc_classification = self.classifier.classify_document(
            #             extracted, slide_classifications
            #         )
            #         # Update result
            #     except Exception as e:
            #         # Handle error
            
            # TODO: Step 5 - Add result to lists
            # results.append(result)
            # self.results.append(result)
            
            pass
        
        # Summary
        self._print_summary(results)
        
        return results
    
    def _prepare_slide_contents(self, extracted: Dict[str, Any], 
                               max_slides: int) -> List[Dict[str, Any]]:
        """Prepare slide contents from extracted data"""
        # TODO: Implement slide content preparation
        # Can use sections or chunk text elements
        return []
    
    def _print_summary(self, results: List[Dict[str, Any]]):
        """Print processing summary"""
        # TODO: Calculate and print summary statistics
        pass

# Create your batch processor
# batch_processor = BatchDocumentProcessor(extractor, classifier)

### Solution for Task 4

In [ ]:
# SOLUTION
class BatchDocumentProcessor:
    """Batch processor for multiple documents"""
    
    def __init__(self, extractor: DoclingContentExtractor, 
                 classifier: LangChainDocumentClassifier):
        """Initialize with extractor and classifier"""
        self.extractor = extractor
        self.classifier = classifier
        self.results = []
        
    def process_documents(self, file_paths: List[str], 
                         max_slides_per_doc: int = 10) -> List[Dict[str, Any]]:
        """Process multiple documents in batch"""
        
        results = []
        total_files = len(file_paths)
        
        print(f"🚀 Starting batch processing of {total_files} documents...")
        
        for idx, file_path in enumerate(file_paths):
            print(f"\n📄 Processing document {idx+1}/{total_files}: {os.path.basename(file_path)}")
            
            result = {
                "file_path": file_path,
                "file_name": os.path.basename(file_path),
                "status": "pending",
                "extraction": None,
                "classification": None,
                "errors": []
            }
            
            try:
                # Step 1: Extract content
                print("  📊 Extracting content...")
                extracted = self.extractor.extract_content(file_path)
                result["extraction"] = extracted
                print(f"    ✅ Extracted: {extracted['statistics']}")
                
                # Step 2: Classify slides (limited for performance)
                print("  🏷️ Classifying content...")
                slide_contents = self._prepare_slide_contents(extracted, max_slides_per_doc)
                slide_classifications = []
                
                for i, slide_content in enumerate(slide_contents):
                    try:
                        classification = self.classifier.classify_slide(slide_content)
                        slide_classifications.append(classification)
                    except Exception as e:
                        print(f"    ⚠️ Error classifying slide {i+1}: {str(e)}")
                        result["errors"].append(f"Slide {i+1} classification error: {str(e)}")
                
                # Step 3: Classify document
                if slide_classifications:
                    try:
                        doc_classification = self.classifier.classify_document(
                            extracted, slide_classifications
                        )
                        result["classification"] = doc_classification
                        result["status"] = "success"
                        print(f"    ✅ Classified as: {doc_classification.primary_theme.value}")
                    except Exception as e:
                        print(f"    ⚠️ Error in document classification: {str(e)}")
                        result["errors"].append(f"Document classification error: {str(e)}")
                        result["status"] = "partial"
                else:
                    result["status"] = "failed"
                    result["errors"].append("No slides could be classified")
                    
            except Exception as e:
                print(f"  ❌ Error processing document: {str(e)}")
                result["status"] = "failed"
                result["errors"].append(f"Processing error: {str(e)}")
            
            results.append(result)
            self.results.append(result)
        
        # Summary
        self._print_summary(results)
        
        return results
    
    def _prepare_slide_contents(self, extracted: Dict[str, Any], 
                               max_slides: int) -> List[Dict[str, Any]]:
        """Prepare slide contents from extracted data"""
        slide_contents = []
        
        # If we have sections, use them as slides
        if extracted.get("sections"):
            for i, section in enumerate(extracted["sections"][:max_slides]):
                slide_content = {
                    "text_elements": [elem for elem in extracted.get("text_elements", [])
                                    if elem.get("level", 0) >= section.get("level", 0)][:5],
                    "tables": extracted.get("tables", [])[i:i+1],
                    "figures": extracted.get("figures", [])[i:i+1],
                    "code_blocks": extracted.get("code_blocks", [])[i:i+1],
                    "formulas": extracted.get("formulas", [])[i:i+1]
                }
                slide_contents.append(slide_content)
        else:
            # Create artificial slides by chunking content
            text_elements = extracted.get("text_elements", [])
            chunk_size = max(1, len(text_elements) // max_slides)
            
            for i in range(0, min(len(text_elements), max_slides * chunk_size), chunk_size):
                slide_content = {
                    "text_elements": text_elements[i:i+chunk_size],
                    "tables": extracted.get("tables", [])[i//chunk_size:i//chunk_size+1],
                    "figures": extracted.get("figures", [])[i//chunk_size:i//chunk_size+1],
                    "code_blocks": [],
                    "formulas": []
                }
                slide_contents.append(slide_content)
        
        return slide_contents[:max_slides]
    
    def _print_summary(self, results: List[Dict[str, Any]]):
        """Print processing summary"""
        total = len(results)
        success = sum(1 for r in results if r["status"] == "success")
        partial = sum(1 for r in results if r["status"] == "partial")
        failed = sum(1 for r in results if r["status"] == "failed")
        
        print("\n" + "="*50)
        print("📊 BATCH PROCESSING SUMMARY")
        print("="*50)
        print(f"Total documents: {total}")
        print(f"✅ Successful: {success}")
        print(f"⚠️  Partial: {partial}")
        print(f"❌ Failed: {failed}")
        print(f"Success rate: {(success/total)*100:.1f}%")
        
        # Show theme distribution
        themes = {}
        for r in results:
            if r.get("classification"):
                theme = r["classification"].primary_theme.value
                themes[theme] = themes.get(theme, 0) + 1
        
        if themes:
            print("\n🏷️ Theme Distribution:")
            for theme, count in sorted(themes.items(), key=lambda x: x[1], reverse=True):
                print(f"  {theme}: {count} documents")
    
    def get_statistics(self) -> pd.DataFrame:
        """Get statistics as DataFrame"""
        stats = []
        for result in self.results:
            if result["status"] in ["success", "partial"] and result.get("extraction"):
                stat = result["extraction"]["statistics"].copy()
                stat["file_name"] = result["file_name"]
                stat["status"] = result["status"]
                if result.get("classification"):
                    stat["primary_theme"] = result["classification"].primary_theme.value
                    stat["complexity"] = result["classification"].overall_complexity
                stats.append(stat)
        
        return pd.DataFrame(stats)

# Create batch processor
batch_processor = BatchDocumentProcessor(extractor, classifier)
print("✅ BatchDocumentProcessor initialized")

## Task 5: Create DuckDB Storage System

### Learning Context
DuckDB is an embedded analytical database perfect for storing and querying document classifications. Benefits:
- No server needed (embedded)
- Excellent analytical query performance
- Support for JSON and array types
- Easy integration with pandas

### Your Task
Create a `DuckLakeStorage` class that:
1. Creates schema for documents, slides, and statistics
2. Stores classification results
3. Provides query methods for analysis
4. Generates theme distribution reports

### Hints
- Create tables with proper relationships (foreign keys)
- Use sequences for auto-incrementing IDs
- Store JSON data for flexible metadata
- Return query results as pandas DataFrames

In [ ]:
class DuckLakeStorage:
    """Store and query classified documents in DuckDB"""
    
    def __init__(self, db_path: str = "document_classifications.duckdb"):
        """Initialize DuckDB connection"""
        self.db_path = db_path
        self.conn = duckdb.connect(db_path)
        self._create_schema()
        
    def _create_schema(self):
        """Create schema for storing classifications"""
        
        # TODO: Step 1 - Create documents table
        # self.conn.execute("""
        #     CREATE TABLE IF NOT EXISTS documents (
        #         id INTEGER PRIMARY KEY,
        #         file_path VARCHAR,
        #         file_name VARCHAR,
        #         document_type VARCHAR,
        #         primary_theme VARCHAR,
        #         target_audience VARCHAR,
        #         overall_complexity INTEGER,
        #         extraction_timestamp TIMESTAMP,
        #         processing_status VARCHAR,
        #         metadata JSON
        #     )
        # """)
        
        # TODO: Step 2 - Create slides table
        # self.conn.execute("""
        #     CREATE TABLE IF NOT EXISTS slides (
        #         id INTEGER PRIMARY KEY,
        #         document_id INTEGER REFERENCES documents(id),
        #         slide_index INTEGER,
        #         slide_type VARCHAR,
        #         # Add other fields...
        #     )
        # """)
        
        # TODO: Step 3 - Create statistics table
        
        # TODO: Step 4 - Create sequences for auto-increment
        # self.conn.execute("CREATE SEQUENCE IF NOT EXISTS doc_id_seq START 1")
        
        print("✅ DuckLake schema created")
    
    def store_document(self, result: Dict[str, Any]) -> int:
        """Store a processed document result"""
        
        # TODO: Check if document was successfully processed
        # if result["status"] not in ["success", "partial"]:
        #     print(f"⚠️ Skipping failed document: {result['file_name']}")
        #     return -1
        
        # TODO: Insert document record
        # doc_id = self.conn.execute("SELECT nextval('doc_id_seq')").fetchone()[0]
        
        # TODO: Insert statistics
        
        # TODO: Insert slide classifications
        
        # TODO: Commit changes
        # self.conn.commit()
        
        return -1  # Return document ID
    
    def query_documents(self, theme: Optional[str] = None, 
                       min_complexity: Optional[int] = None) -> pd.DataFrame:
        """Query stored documents"""
        
        # TODO: Build query with optional filters
        # query = """
        #     SELECT 
        #         d.file_name,
        #         d.document_type,
        #         d.primary_theme,
        #         d.target_audience,
        #         d.overall_complexity
        #     FROM documents d
        #     WHERE 1=1
        # """
        
        # TODO: Add filters
        # params = []
        # if theme:
        #     query += " AND d.primary_theme = ?"
        #     params.append(theme)
        
        # TODO: Execute and return DataFrame
        # return self.conn.execute(query, params).df()
        
        return pd.DataFrame()  # Empty DataFrame for now
    
    def close(self):
        """Close database connection"""
        self.conn.close()

# Create your storage instance
# storage = DuckLakeStorage()

### Solution for Task 5

In [ ]:
# SOLUTION
class DuckLakeStorage:
    """Store and query classified documents in DuckDB"""
    
    def __init__(self, db_path: str = "document_classifications.duckdb"):
        """Initialize DuckDB connection"""
        self.db_path = db_path
        self.conn = duckdb.connect(db_path)
        self._create_schema()
        
    def _create_schema(self):
        """Create schema for storing classifications"""
        
        # Documents table
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                id INTEGER PRIMARY KEY,
                file_path VARCHAR,
                file_name VARCHAR,
                document_type VARCHAR,
                primary_theme VARCHAR,
                target_audience VARCHAR,
                overall_complexity INTEGER,
                extraction_timestamp TIMESTAMP,
                processing_status VARCHAR,
                metadata JSON
            )
        """)
        
        # Slides/sections table
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS slides (
                id INTEGER PRIMARY KEY,
                document_id INTEGER REFERENCES documents(id),
                slide_index INTEGER,
                slide_type VARCHAR,
                content_theme VARCHAR,
                visual_density VARCHAR,
                complexity_score INTEGER,
                confidence_score FLOAT,
                has_code BOOLEAN,
                has_formulas BOOLEAN,
                has_tables BOOLEAN,
                has_figures BOOLEAN,
                key_topics VARCHAR[]
            )
        """)
        
        # Content elements table
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS content_elements (
                id INTEGER PRIMARY KEY,
                document_id INTEGER REFERENCES documents(id),
                element_type VARCHAR,
                content TEXT,
                element_level INTEGER,
                metadata JSON
            )
        """)
        
        # Statistics table
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS document_statistics (
                document_id INTEGER PRIMARY KEY REFERENCES documents(id),
                total_text_elements INTEGER,
                total_tables INTEGER,
                total_figures INTEGER,
                total_code_blocks INTEGER,
                total_formulas INTEGER,
                total_lists INTEGER,
                total_sections INTEGER
            )
        """)
        
        # Create sequences
        self.conn.execute("CREATE SEQUENCE IF NOT EXISTS doc_id_seq START 1")
        self.conn.execute("CREATE SEQUENCE IF NOT EXISTS slide_id_seq START 1")
        self.conn.execute("CREATE SEQUENCE IF NOT EXISTS element_id_seq START 1")
        
        print("✅ DuckLake schema created")
    
    def store_document(self, result: Dict[str, Any]) -> int:
        """Store a processed document result"""
        
        if result["status"] not in ["success", "partial"]:
            print(f"⚠️ Skipping failed document: {result['file_name']}")
            return -1
        
        extraction = result["extraction"]
        classification = result.get("classification")
        
        # Insert document
        doc_id = self.conn.execute("SELECT nextval('doc_id_seq')").fetchone()[0]
        
        self.conn.execute("""
            INSERT INTO documents (
                id, file_path, file_name, document_type, primary_theme,
                target_audience, overall_complexity, extraction_timestamp,
                processing_status, metadata
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            doc_id,
            result["file_path"],
            result["file_name"],
            classification.document_type if classification else "unknown",
            classification.primary_theme.value if classification else "unknown",
            classification.target_audience if classification else "unknown",
            classification.overall_complexity if classification else 0,
            extraction["metadata"].get("extraction_timestamp"),
            result["status"],
            json.dumps(extraction["metadata"])
        ])
        
        # Insert statistics
        stats = extraction["statistics"]
        self.conn.execute("""
            INSERT INTO document_statistics VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            doc_id,
            stats["total_text_elements"],
            stats["total_tables"],
            stats["total_figures"],
            stats["total_code_blocks"],
            stats["total_formulas"],
            stats["total_lists"],
            stats["total_sections"]
        ])
        
        # Insert slide classifications
        if classification and classification.slide_classifications:
            for idx, slide_class in enumerate(classification.slide_classifications):
                slide_id = self.conn.execute("SELECT nextval('slide_id_seq')").fetchone()[0]
                
                self.conn.execute("""
                    INSERT INTO slides (
                        id, document_id, slide_index, slide_type, content_theme,
                        visual_density, complexity_score, confidence_score,
                        has_code, has_formulas, has_tables, has_figures, key_topics
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, [
                    slide_id,
                    doc_id,
                    idx,
                    slide_class.slide_type.value,
                    slide_class.content_theme.value,
                    slide_class.visual_density.value,
                    slide_class.complexity_score,
                    slide_class.confidence_score,
                    slide_class.has_code,
                    slide_class.has_formulas,
                    slide_class.has_tables,
                    slide_class.has_figures,
                    slide_class.key_topics
                ])
        
        # Insert sample content elements (limit for performance)
        for elem_type, elements in [
            ("text", extraction.get("text_elements", [])[:10]),
            ("section", extraction.get("sections", [])[:10])
        ]:
            for elem in elements:
                elem_id = self.conn.execute("SELECT nextval('element_id_seq')").fetchone()[0]
                
                self.conn.execute("""
                    INSERT INTO content_elements (
                        id, document_id, element_type, content, element_level, metadata
                    ) VALUES (?, ?, ?, ?, ?, ?)
                """, [
                    elem_id,
                    doc_id,
                    elem_type,
                    elem.get("text", elem.get("title", "")),
                    elem.get("level", 0),
                    json.dumps({})
                ])
        
        self.conn.commit()
        print(f"✅ Stored document: {result['file_name']} (ID: {doc_id})")
        return doc_id
    
    def query_documents(self, theme: Optional[str] = None, 
                       min_complexity: Optional[int] = None) -> pd.DataFrame:
        """Query stored documents"""
        
        query = """
            SELECT 
                d.file_name,
                d.document_type,
                d.primary_theme,
                d.target_audience,
                d.overall_complexity,
                s.total_text_elements,
                s.total_tables,
                s.total_figures
            FROM documents d
            JOIN document_statistics s ON d.id = s.document_id
            WHERE 1=1
        """
        
        params = []
        if theme:
            query += " AND d.primary_theme = ?"
            params.append(theme)
        if min_complexity:
            query += " AND d.overall_complexity >= ?"
            params.append(min_complexity)
        
        return self.conn.execute(query, params).df()
    
    def get_theme_distribution(self) -> pd.DataFrame:
        """Get distribution of themes"""
        
        return self.conn.execute("""
            SELECT 
                primary_theme,
                COUNT(*) as document_count,
                AVG(overall_complexity) as avg_complexity,
                SUM(s.total_text_elements) as total_text_elements
            FROM documents d
            JOIN document_statistics s ON d.id = s.document_id
            GROUP BY primary_theme
            ORDER BY document_count DESC
        """).df()
    
    def close(self):
        """Close database connection"""
        self.conn.close()

# Create storage instance
storage = DuckLakeStorage()
print("✅ DuckLakeStorage initialized")

## Challenge Task: Complete Document Processing Pipeline

### Your Challenge
Now that you've built all the components, create a complete pipeline that:

1. Finds documents in a directory
2. Processes them in batch
3. Stores results in DuckDB
4. Generates analysis reports
5. Evaluates classification accuracy

### Requirements
- Handle different document formats (PDF, PPTX, DOCX)
- Provide progress updates
- Generate visualizations of results
- Export results to CSV for further analysis

### Starter Code
Build your solution below:

In [ ]:
async def document_classification_pipeline(
    input_directory: str,
    output_directory: str = "./classification_results",
    file_extensions: List[str] = [".pdf", ".pptx", ".docx"],
    max_files: Optional[int] = None
):
    """
    Complete document classification pipeline.
    
    Args:
        input_directory: Directory containing documents
        output_directory: Directory for results
        file_extensions: File types to process
        max_files: Maximum number of files to process
    
    Returns:
        Summary report of classification results
    """
    # TODO: Your implementation here
    # Steps:
    # 1. Find all documents in directory
    # 2. Process them using batch processor
    # 3. Store results in DuckDB
    # 4. Generate analysis reports
    # 5. Export results
    
    pass

# Test your pipeline
# report = await document_classification_pipeline(
#     input_directory="/path/to/documents",
#     max_files=10
# )

### Solution for Challenge Task

In [ ]:
# SOLUTION
async def document_classification_pipeline(
    input_directory: str,
    output_directory: str = "./classification_results",
    file_extensions: List[str] = [".pdf", ".pptx", ".docx"],
    max_files: Optional[int] = None
):
    """
    Complete document classification pipeline.
    """
    import os
    from datetime import datetime
    
    # Create output directory
    os.makedirs(output_directory, exist_ok=True)
    
    # Step 1: Find documents
    print(f"🔍 Searching for documents in: {input_directory}")
    documents = []
    
    for root, dirs, files in os.walk(input_directory):
        for file in files:
            if any(file.lower().endswith(ext) for ext in file_extensions):
                documents.append(os.path.join(root, file))
    
    if max_files:
        documents = documents[:max_files]
    
    print(f"📄 Found {len(documents)} documents to process")
    
    if not documents:
        return {"error": "No documents found"}
    
    # Step 2: Process documents
    print("\n🚀 Starting classification pipeline...")
    
    # Initialize components
    extractor = DoclingContentExtractor()
    classifier = LangChainDocumentClassifier()
    processor = BatchDocumentProcessor(extractor, classifier)
    storage = DuckLakeStorage(os.path.join(output_directory, "classifications.duckdb"))
    
    # Process documents
    results = processor.process_documents(documents, max_slides_per_doc=5)
    
    # Step 3: Store results
    print("\n💾 Storing results in database...")
    stored_count = 0
    for result in results:
        doc_id = storage.store_document(result)
        if doc_id > 0:
            stored_count += 1
    
    # Step 4: Generate analysis
    print("\n📊 Generating analysis reports...")
    
    # Get theme distribution
    theme_dist = storage.get_theme_distribution()
    if not theme_dist.empty:
        theme_dist.to_csv(os.path.join(output_directory, "theme_distribution.csv"), index=False)
        print("✅ Saved theme distribution to theme_distribution.csv")
    
    # Get all documents
    all_docs = storage.query_documents()
    if not all_docs.empty:
        all_docs.to_csv(os.path.join(output_directory, "all_documents.csv"), index=False)
        print("✅ Saved document list to all_documents.csv")
    
    # Get statistics
    stats_df = processor.get_statistics()
    if not stats_df.empty:
        stats_df.to_csv(os.path.join(output_directory, "document_statistics.csv"), index=False)
        print("✅ Saved statistics to document_statistics.csv")
    
    # Step 5: Create summary report
    report = {
        "pipeline_run": datetime.now().isoformat(),
        "input_directory": input_directory,
        "output_directory": output_directory,
        "documents_found": len(documents),
        "documents_processed": len(results),
        "documents_stored": stored_count,
        "success_rate": sum(1 for r in results if r["status"] == "success") / len(results) if results else 0,
        "theme_distribution": theme_dist.to_dict('records') if not theme_dist.empty else [],
        "output_files": [
            "classifications.duckdb",
            "theme_distribution.csv",
            "all_documents.csv",
            "document_statistics.csv",
            "pipeline_report.json"
        ]
    }
    
    # Save report
    with open(os.path.join(output_directory, "pipeline_report.json"), 'w') as f:
        json.dump(report, f, indent=2)
    
    # Create visualization (optional)
    try:
        import matplotlib.pyplot as plt
        
        if not theme_dist.empty:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            
            # Theme distribution
            theme_dist.plot(x='primary_theme', y='document_count', kind='bar', ax=ax1)
            ax1.set_title('Document Distribution by Theme')
            ax1.set_xlabel('Theme')
            ax1.set_ylabel('Number of Documents')
            
            # Complexity distribution
            theme_dist.plot(x='primary_theme', y='avg_complexity', kind='bar', ax=ax2)
            ax2.set_title('Average Complexity by Theme')
            ax2.set_xlabel('Theme')
            ax2.set_ylabel('Average Complexity (1-10)')
            
            plt.tight_layout()
            plt.savefig(os.path.join(output_directory, "classification_analysis.png"))
            print("✅ Saved visualization to classification_analysis.png")
            plt.close()
    except ImportError:
        print("⚠️ Matplotlib not available, skipping visualization")
    
    # Cleanup
    storage.close()
    
    print(f"\n✅ Pipeline complete! Results saved to: {output_directory}")
    return report

# Example usage:
# report = await document_classification_pipeline(
#     input_directory="/Users/daviddrummond/coursera/raw ppt files",
#     max_files=5
# )
# print(json.dumps(report, indent=2))

## Summary & Next Steps

### What You've Learned
✅ Extract and categorize content with Docling  
✅ Build structured classification pipelines with LangChain  
✅ Define type-safe schemas with Pydantic  
✅ Implement batch processing with error handling  
✅ Store and query results with DuckDB  
✅ Evaluate classification accuracy  

### Key Concepts Mastered
1. **Document Intelligence**: Automatic extraction of structured content
2. **Structured LLM Outputs**: Forcing LLMs to return validated data
3. **Scalable Processing**: Handling multiple documents efficiently
4. **Analytics Storage**: Using DuckDB for fast queries
5. **Quality Metrics**: Measuring and improving accuracy

### Practice Exercises
1. **Add Custom Classifications**: Create new classification schemas for specific domains
2. **Improve Accuracy**: Fine-tune prompts for better classification
3. **Add Embeddings**: Generate embeddings for semantic search
4. **Build API**: Create a REST API for classification services
5. **Create Dashboard**: Build a web dashboard for results

### Next Steps
- **Demo 3**: Use these classifications for RAG applications
- **Demo 4**: Build search interfaces over classified content
- **Production**: Scale to thousands of documents
- **Integration**: Connect with existing document management systems

### Resources
- [Docling Documentation](https://github.com/DS4SD/docling)
- [LangChain Structured Output](https://python.langchain.com/docs/modules/model_io/output_parsers/structured)
- [Pydantic Documentation](https://docs.pydantic.dev/)
- [DuckDB Documentation](https://duckdb.org/docs/)

Happy classifying! 🎯